# Example 7.5.1 / 1: Choose Between Two Regions

Companion notebook for **7.5.1 and, or, not and Comparisons in Constraints** in the AMPL lab Documentation, section "Choose Between Two Regions". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Choose Between Two Regions

The point `(x, y)` must lie in one of two separate regions, and must also satisfy
`x + y >= 6`. The objective prefers small `x` and `y`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x integer >= 0, <= 10;        # first coordinate
var y integer >= 0, <= 10;        # second coordinate

# (x, y) lies in one of two separate regions
subject to EitherRegion:
    (x <= 2 and y <= 4) or (x >= 7 and y >= 6);
subject to Enough: x + y >= 6;    # rules out most of the first region
minimize Cost: 3 * x + 2 * y;

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.solve(solver="highs")
print("x =", ampl.var["x"].value(), " y =", ampl.var["y"].value())

HiGHS 1.14.0: optimal solution; objective 14
0 simplex iterations
0 branching nodes
x = 2.0  y = 4.0


### Pure AMPL: model.run

Only the corner `(2, 4)` of the first region meets `x + y >= 6`, and it costs 14, against 33 at
the cheapest point `(7, 6)` of the second region. Points between the regions, such as `(4, 2)`,
are excluded even though they would satisfy `Enough`.

The same in pure AMPL, with the same `model.mod`:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option solver highs;
solve;
display x, y;

HiGHS 1.14.0: optimal solution; objective 14
0 simplex iterations
0 branching nodes
x = 2
y = 4

